In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_csv("hf://datasets/AzharAli05/Resume-Screening-Dataset/dataset.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
df

,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
...,...,...,...,...,...
10169,Product Manager,Here's a sample resume for Diana Miller:\n\n**...,reject,Unsatisfactory references or background check.,Here is a comprehensive job description for a ...
10170,UI Engineer,Here's a sample resume for Grace Taylor:\n\n**...,reject,Lack of relevant skills or experience.,Here is a sample job description for a UI Engi...
10171,UI Engineer,Here's a sample resume for Hank Brown:\n\n**Ha...,select,Growth mindset and adaptability.,Here is a job description for a UI Engineer ro...
10172,Data Engineer,Here's a sample resume for Diana Wilson:\n\n**...,reject,Lack of relevant skills or experience.,Here is a comprehensive job description for a ...


In [3]:
df.describe()

,Role,Resume,Decision,Reason_for_decision,Job_Description
count,10174,10174,10174,10174,10174
unique,45,10174,2,539,3446
top,Data Scientist,"Here's a sample resume for Charlie Miller, a P...",reject,Insufficient system design expertise for senio...,Join our team as a Product Manager and leverag...
freq,538,1,5114,730,103


In [4]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,]', '', text)
    return text.lower().strip()

df['clean_resume'] = df['Resume'].apply(clean_text)
df['clean_jd'] = df['Job_Description'].apply(clean_text)

df[['clean_resume', 'clean_jd']].head(2)


,clean_resume,clean_jd
0,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...
1,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...


In [5]:
label_map = {
    "select": 1,
    "reject": 0
}

df['label'] = df['Decision'].map(label_map)

df[['Decision', 'label']].head()

,Decision,label
0,reject,0
1,select,1
2,reject,0
3,select,1
4,reject,0


In [6]:
df['label'].value_counts()


,count
label,
0,5114
1,5060


In [7]:
df.head()

,Role,Resume,Decision,Reason_for_decision,Job_Description,clean_resume,clean_jd,label
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...,0
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...,1
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...,heres a professional resume for patrick mcclai...,we need a human resources specialist to enhanc...,0
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...,heres a professional resume for patricia gray ...,be part of a passionate team at the forefront ...,1
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...,heres a professional resume for amanda gross a...,we are looking for an experienced ecommerce sp...,0


In [8]:
def extract_section(text, keywords, max_len=400):
    for key in keywords:
        if key in text:
            start = text.find(key)
            section = text[start:start + max_len]
            return section
    return ""

skills_keys =  [
    'skills',
    'technical skills',
    'core skills',
    'key skills',
    'technologies'
]
exp_keys =[
    'professional experience',
    'work experience',
    'experience',
    'employment history'
]
edu_keys =  [
    'education',
    'academic background',
    'qualifications',
    'degree'
]
cert_keys = ['certification', 'certifications']

df['skills_text'] = df['clean_resume'].apply(lambda x: extract_section(x, skills_keys))
df['experience_text'] = df['clean_resume'].apply(lambda x: extract_section(x, exp_keys))
df['education_text'] = df['clean_resume'].apply(lambda x: extract_section(x, edu_keys))
df['certificate_text']=df['clean_resume'].apply(lambda x:extract_section(x,cert_keys))
df.head()


,Role,Resume,Decision,Reason_for_decision,Job_Description,clean_resume,clean_jd,label,skills_text,experience_text,education_text,certificate_text
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...,0,skills inventory management seo for ecommerc...,"professional experience ecommerce specialist, ...",education bachelors degree in business admini...,certifications google analytics certification...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...,1,,experience in creating engaging game experienc...,,
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...,heres a professional resume for patrick mcclai...,we need a human resources specialist to enhanc...,0,"skills hris systems workday, bamboohr, etc. ...","work experience human resources generalist, xy...",education bachelors degree in human resources...,certifications shrmcp society for human resou...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...,heres a professional resume for patricia gray ...,be part of a passionate team at the forefront ...,1,skills product listing management seo for ec...,"work experience ecommerce specialist, abc onli...",education bachelors degree in business admini...,certification program certifications google a...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...,heres a professional resume for amanda gross a...,we are looking for an experienced ecommerce sp...,0,skills inventory management software tradege...,"work experience ecommerce specialist, abc onli...",education bachelors degree in business admini...,certifications google analytics certification...


In [9]:
print(df.iloc[1]["Resume"])


Here's a professional resume for Ann Marshall:

Ann Marshall
Contact Information:

* Email: [ann.marshall@email.com](mailto:ann.marshall@email.com)
* Phone: (123) 456-7890
* LinkedIn: linkedin.com/in/annmarshall
* GitHub: github.com/annmarshall

Summary:
Highly motivated and detail-oriented game developer with 5+ years of experience in creating engaging game experiences. Skilled in Unity and Unreal Engine, with expertise in C


In [10]:
training_data = []

for _, row in df.iterrows():
    for section in ['skills_text', 'experience_text', 'education_text','certificate_text']:
        if row[section].strip() != "":
            training_data.append({
                "text_a": row[section],
                "text_b": row['clean_jd'],
                "label": row['label']
            })

train_df = pd.DataFrame(training_data)

print(train_df.head())
print("Total training samples:", len(train_df))


                                              text_a  \
0  skills  inventory management  seo for ecommerc...   
1  professional experience ecommerce specialist, ...   
2  education  bachelors degree in business admini...   
3  certifications  google analytics certification...   
4  experience in creating engaging game experienc...   

                                              text_b  label  
0  be part of a passionate team at the forefront ...      0  
1  be part of a passionate team at the forefront ...      0  
2  be part of a passionate team at the forefront ...      0  
3  be part of a passionate team at the forefront ...      0  
4  help us build the nextgeneration products as a...      1  
Total training samples: 37632


In [11]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    list(zip(train_df['text_a'], train_df['text_b'])),
    train_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)


In [12]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_pairs(text_pairs):
    return tokenizer(
        [a for a, b in text_pairs],
        [b for a, b in text_pairs],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_pairs(train_texts)
val_encodings = tokenize_pairs(val_texts)


In [13]:
train_encodings

{'input_ids': tensor([[  101,  2658,  3325,  ...,     0,     0,     0],
        [  101, 10618,  2015,  ...,     0,     0,     0],
        [  101,  2658,  3325,  ...,     0,     0,     0],
        ...,
        [  101,  3325,  1999,  ...,  8145,  1010,   102],
        [  101, 10618,  2015,  ...,     0,     0,     0],
        [  101,  4813,  4730,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [14]:
import torch

class ResumeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ResumeDataset(train_encodings, train_labels)
val_dataset = ResumeDataset(val_encodings, val_labels)


In [15]:
train_dataset[0]

{'input_ids': tensor([  101,  2658,  3325, 21331,  3992,  1010,  1060,  2100,  2480, 21331,
          2760, 28994,  4765,  2881,  1998,  2764, 20478,  3001,  2005,  2536,
          5097,  1010,  2164,  8392,  9163,  1010, 20854,  1010,  1998, 16924,
         12550, 20996,  2015,  8957,  4082,  2291,  2000, 17409,  1998,  2491,
         20478,  3001,  1010,  1998,  7528,  2033,  7507, 15312,  2594,  6177,
          2107,  2004, 13907,  1010,  2552,  6692,  6591,  1010,  1998,  2491,
          3001,  8678,  2007,  2892, 11263, 27989,  2389,  2780,   102,  3693,
          2256,  2136,  2004,  1037,  4031,  3208,  1998, 21155,  2115,  1020,
          2086,  1997,  3325,  2000,  2191,  2019,  4254,  1998,  9002,  2000,
          9525,  3934,  1012,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [16]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2   # 0 = reject, 1 = select
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
sample = train_dataset[0]

outputs = model(
    input_ids=sample['input_ids'].unsqueeze(0),
    attention_mask=sample['attention_mask'].unsqueeze(0),
    labels=sample['labels'].unsqueeze(0)
)

print(outputs)


SequenceClassifierOutput(loss=tensor(0.5331, grad_fn=<NllLossBackward0>), logits=tensor([[-0.1493, -0.5000]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [18]:
torch.argmax(outputs.logits, dim=1).item()


0

In [19]:
!pip install -U transformers accelerate


In [20]:
import transformers
print(transformers.__version__)


5.1.0


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
trainer.evaluate()

In [ ]:
device = model.device


In [ ]:
sample = train_dataset[0]

inputs = {
    'input_ids': sample['input_ids'].unsqueeze(0).to(device),
    'attention_mask': sample['attention_mask'].unsqueeze(0).to(device),
    'labels': sample['labels'].unsqueeze(0).to(device)
}

outputs = model(**inputs)
print(outputs.logits)


In [ ]:
import torch

probs = torch.softmax(outputs.logits, dim=1)
print(probs)


In [ ]:
predicted_class = torch.argmax(probs, dim=1).item()

if predicted_class == 1:
    print("✅ Resume Selected")
else:
    print("❌ Resume Rejected")


In [ ]:
predictions = trainer.predict(val_dataset)


In [ ]:
import torch

logits = torch.tensor(predictions.predictions)
labels = torch.tensor(predictions.label_ids)


In [ ]:
probs = torch.softmax(logits, dim=1)
predicted_classes = torch.argmax(probs, dim=1)
for i in range(10):
    print(
        f"Pred: {predicted_classes[i].item()}, "
        f"True: {labels[i].item()}, "
        f"Prob(Selected): {probs[i][1]:.3f}"
    )


In [ ]:
def predict_match(section_text, job_description):
    inputs = tokenizer(
        section_text,
        job_description,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    return probs[0][1].item()  # probability of "Selected"


In [ ]:
def resume_job_match(resume_sections, job_description, mode="avg"):
    scores = []

    for section in resume_sections:
        if section.strip():
            score = predict_match(section, job_description)
            scores.append(score)

    if not scores:
        return 0.0

    return max(scores) if mode == "max" else sum(scores) / len(scores)


In [ ]:
row = df.iloc[1]   # any index you want
skills_text = row['skills_text']
experience_text = row['experience_text']
education_text = row['education_text']



In [ ]:
skills_text = """
Customer Service, Sales, Retail Operations
"""

experience_text = """
Worked in offline retail sales and handled customer interactions.
"""

education_text = """
Diploma in Retail Management.
"""


In [ ]:


jd_text = "We are looking for an E-commerce Specialist with strong experience in inventory management, SEO optimization, online advertising, and data analytics.The candidate should have hands-on experience with Shopify or WooCommerce,Google Analytics, and digital marketing tools."


In [ ]:

final_score = resume_job_match(
    resume_sections=[
        skills_text,
        experience_text,
        education_text
    ],
    job_description=jd_text,
    mode="avg"   # or "max"
)

decision = "SELECTED" if final_score > 0.5 else "REJECTED"

print("Final Score:", final_score)
print("Decision:", decision)
